# Tractable Computation of the Minimal-Weakening Reason on SDDs




| Section | Content |
|---|---|
| 1 | Setup and implementation of the algorithms |
| 2 | Result 1: bottom-up distance computation |
| 3 | Result 2: backpointer reconstruction |
| 4 | The full pipeline on wine classifier |
| 5 | Result 3: minimality survives ties |
| 6 | Randomised stress test of Results 1 to 3 |
| 7 | Result 4: a bounded menu of alternative reasons |
| 8 | Scaling: why enumeration is not an option |


## 1. Setup and implementation



In [1]:
!pip install pysdd --quiet



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\sofia\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [20]:
from pysdd.sdd import SddManager, Vtree
from itertools import product
import math, random, time

INF = math.inf

def build(nvars, var_order=None, vtree_type="right"):
    # creating a fresh SDD manager
    vt = Vtree(var_count=nvars, var_order=var_order or list(range(1, nvars + 1)), vtree_type=vtree_type)
    mgr = SddManager.from_vtree(vt)
    return mgr, [mgr.literal(i) for i in range(1, nvars + 1)] # returning manager and list of positive literal nodes

def fmt(term, names):
    #making a term readable
    if not term:
        return "T"
    return " & ".join(names[v] if val else "!" + names[v] for v, val in sorted(term.items()))

def disagreement(term, omega):
    # number of literals in `term` that conflict with instance `omega`
    return sum(1 for v, val in term.items() if omega[v] != val)




### 1.1 The distance recursion (Result 1)




In [ ]:
def node_dist(node, omega, memo):
    """ bottom-up minimum Hamming distance from omega to nearest model of `node`"""
    key = node.id
    if key in memo:
        return memo[key]

    if node.is_false(): # no models at all
        r = INF
    elif node.is_true(): # trivially satisfied, nothing to flip
        r = 0
    elif node.is_literal(): # base case: match or one flip
        lit = node.literal
        r = 0 if omega[abs(lit)] == (lit > 0) else 1
    else: # decomposition: min over branches, sum within a branch
        r = INF
        for prime, sub in node.elements():
            r = min(r, node_dist(prime, omega, memo) + node_dist(sub, omega, memo))

    memo[key] = r
    return r


### 1.2 Backpointer reconstruction (Result 2)


In [ ]:
def node_reconstruct(node, omega, memo):

    if node.is_false():
        raise ValueError("unsatisfiable node")
    if node.is_true():
        return {}  # constrains nothing
    if node.is_literal():
        lit = node.literal
        return {abs(lit): lit > 0}

    # in the case of decomposition, we  use recursion to find the winning branch and combine the assignments from its prime and sub nodes
    best, best_cost = None, INF
    for prime, sub in node.elements(): # find the branch that won
        c = node_dist(prime, omega, memo) + node_dist(sub, omega, memo)
        if c < best_cost:
            best_cost, best = c, (prime, sub)

    prime, sub = best
    assignment = dict(node_reconstruct(prime, omega, memo))
    assignment.update(node_reconstruct(sub, omega, memo))
    return assignment


def nearest_model(node, omega, nvars):
    memo = {}
    d = node_dist(node, omega, memo)
    if d == INF:
        return None, INF
    partial = node_reconstruct(node, omega, memo)
    # free variables (unmentioned by the winning branches) cost nothing: take omega's value
    full = {v: partial.get(v, omega[v]) for v in range(1, nvars + 1)}
    return full, d


### 1.3 Greedy shrinking


In [ ]:
def is_implicant(term, node, mgr):
    """Does `term` (dict var -> bool) imply `node`?  True iff term & !node is unsatisfiable"""
    t = mgr.true()
    for v, val in term.items():
        t = t & (mgr.literal(v) if val else mgr.literal(-v))
    return (t & ~node).is_false()


def greedy_shrink(model, node, mgr, order=None):
    """Full pipeline step 2: shrink a model down to a prime implicant by dropping literals"""
    term = dict(model)
    for v in (order or list(model.keys())):
        if v not in term:
            continue
        trial = {k: val for k, val in term.items() if k != v}
        if is_implicant(trial, node, mgr):  # still sufficient without v? then drop it
            term = trial
    return term


### 1.4 Brute-force ground truth




In [6]:
def brute_nearest(node, omega, nvars, mgr):
    """Ground truth for Results 1-2: enumerate every assignment, keep the closest model."""
    best, best_d = None, INF
    for bits in product([False, True], repeat=nvars):
        asg = {v: bits[v - 1] for v in range(1, nvars + 1)}
        if not is_implicant(asg, node, mgr):     # a full assignment implies node iff it is a model
            continue
        d = sum(1 for v in range(1, nvars + 1) if asg[v] != omega[v])
        if d < best_d:
            best_d, best = d, asg
    return best, best_d


def all_prime_implicants(node, nvars, mgr):
    """Ground truth for Result 3: every prime implicant, by exhaustive 3^n search."""
    out, seen = [], set()
    for vals in product([None, True, False], repeat=nvars):
        term = {i + 1: vals[i] for i in range(nvars) if vals[i] is not None}
        if not term and not node.is_true():
            continue
        if not is_implicant(term, node, mgr):
            continue
        # prime = no proper subset is still an implicant
        if any(is_implicant({k: x for k, x in term.items() if k != v}, node, mgr) for v in term):
            continue
        key = tuple(sorted(term.items()))
        if key not in seen:
            seen.add(key)
            out.append(term)
    return out


## 2. Result 1: the distance can be computed bottom-up




In [ ]:
mgr, (a, b, c) = build(3)
phi = (a & b) | c
names = {1: "a", 2: "b", 3: "c"}
omega = {1: False, 2: False, 3: False}

print(f"phi is satisfied by omega?  {is_implicant(omega, phi, mgr)}   <- the reason an edit is needed")
print(f"SDD size (nodes):           {phi.size()}")
print(f"model count:                {phi.global_model_count()}")

# every model of phi, with its distance to omega - the thing we are trying to minimise
print("\nAll models of phi and their Hamming distance to omega:")
for bits in product([False, True], repeat=3):
    asg = {v: bits[v - 1] for v in [1, 2, 3]}
    if is_implicant(asg, phi, mgr):
        d = sum(1 for v in [1, 2, 3] if asg[v] != omega[v])
        print(f"   a={int(asg[1])} b={int(asg[2])} c={int(asg[3])}   distance {d}")


phi is satisfied by omega?  0   <- the reason an edit is needed
SDD size (nodes):           4
model count:                5

All models of phi and their Hamming distance to omega:
   a=0 b=0 c=1   distance 1
   a=0 b=1 c=1   distance 2
   a=1 b=0 c=1   distance 2
   a=1 b=1 c=0   distance 2
   a=1 b=1 c=1   distance 3


In [8]:
memo = {}
d_algo = node_dist(phi, omega, memo)
_, d_brute = brute_nearest(phi, omega, 3, mgr)

print(f"bottom-up recursion : dist = {d_algo}")
print(f"brute-force check   : dist = {d_brute}")
print(f"MATCH: {d_algo == d_brute}")
print(f"\nnodes visited (memo entries): {len(memo)}  <- each computed exactly once, despite sharing")


bottom-up recursion : dist = 1
brute-force check   : dist = 1
MATCH: True

nodes visited (memo entries): 8  <- each computed exactly once, despite sharing


## 3. Result 2: recovering the actual nearest instance




In [ ]:
full, d = nearest_model(phi, omega, 3)
brute_full, brute_d = brute_nearest(phi, omega, 3, mgr)

print(f"reconstructed omega_0: {full}   distance {d}")
print(f"brute-force nearest: {brute_full}   distance {brute_d}")
print()
print(f"is omega_0 actually a model of phi?    {is_implicant(full, phi, mgr)}")
print(f"is its distance to omega exactly dist?     " f"{sum(1 for v in full if full[v] != omega[v]) == d}")
print(f"\n(b was unconstrained on the winning branch and was filled from omega: b={full[2]})")


reconstructed omega_0 : {1: False, 2: False, 3: True}   distance 1
brute-force nearest   : {1: False, 2: False, 3: True}   distance 1

is omega_0 actually a model of phi?        1
is its distance to omega exactly dist?     True

(b was unconstrained on the winning branch and was filled from omega: b=False)


## 4. The full pipeline on wine classifier




In [ ]:
wmgr, (M, R, F, W) = build(4)
delta = (~M) & (~F | W) & (W | R)
wnames = {1: "m", 2: "r", 3: "f", 4: "w"}
womega = {1: True, 2: True, 3: True, 4: True}

# STEP 1: nearest model 
w0, wd = nearest_model(delta, womega, 4)
print(f"step 1  nearest model omega_0 = {w0}, distance {wd}")
print(f"        brute-force check = {brute_nearest(delta, womega, 4, wmgr)[1]}")

# STEP 2: greedy shrink from it
tau = greedy_shrink(w0, delta, wmgr)
print(f"step 2  greedy shrink   -> reason: {fmt(tau, wnames)}")
print(f"        disagreement with omega: {disagreement(tau, womega)}")

# CHECK against the full (exponential) set of reasons the naive method would need
print("\nground truth - every prime implicant of Delta:")
wpis = all_prime_implicants(delta, 4, wmgr)
for p in wpis:
    print(f"   {fmt(p, wnames):20s} disagreement {disagreement(p, womega)}")
best = min(disagreement(p, womega) for p in wpis)
print(f"\nminimum disagreement over ALL reasons: {best}")
print(f"pipeline achieved it WITHOUT enumerating them: {disagreement(tau, womega) == best}")


step 1  nearest model omega_0 = {1: False, 2: True, 3: True, 4: True}, distance 1
        brute-force check      = 1
step 2  greedy shrink          -> reason: !m & w
        disagreement with omega: 1

ground truth -- every prime implicant of Delta:
   !m & w               disagreement 1
   !m & r & !f          disagreement 2

minimum disagreement over ALL reasons: 1
pipeline achieved it WITHOUT enumerating them: True


In [ ]:
# STEP 3: WEAKEN the reason by dropping exactly the literals that disagree with omega
weak_term = {v: val for v, val in tau.items() if womega[v] == val}

print(f"selected reason : {fmt(tau, wnames)}")
print(f"literals disagreeing with omega: "
      f"{[wnames[v] for v, val in tau.items() if womega[v] != val]}   <- these get dropped")
print(f"WEAKENED reason : {fmt(weak_term, wnames)}")

# STEP 4: build the edited classifier.  we need only Delta OR (weakened reason). the untouched reasons never have to be listed.
weakened = wmgr.true()
for v, val in weak_term.items():
    weakened = weakened & (wmgr.literal(v) if val else wmgr.literal(-v))

delta_edited = delta | weakened

print()
print(f"models before edit : {delta.global_model_count()}")
print(f"models after edit  : {delta_edited.global_model_count()}")
print()
print(f"is omega now classified positively? {bool(is_implicant(womega, delta_edited, wmgr))}   <- the edit worked")
print(f"did we keep everything already positive? {bool((delta & ~delta_edited).is_false())}   <- nothing was lost")


selected reason        : !m & w
literals disagreeing with omega: ['m']   <- these get dropped
WEAKENED reason        : w

models before edit : 5
models after edit  : 9

is omega now classified positively?      True   <- the edit worked
did we keep everything already positive? True   <- nothing was lost




## 5. Result 3: minimality survives ties




In [ ]:
# TIE CASE 1: XNOR - two models, both at distance 1, perfectly symmetric
tmgr, (X, Y) = build(2)
xnor = (X & Y) | (~X & ~Y)
tnames = {1: "x", 2: "y"}
tomega = {1: True, 2: False}

print("Delta = (x & y) | (!x & !y),  omega = (x=T, y=F)")
print(f"nearest distance: {nearest_model(xnor, tomega, 2)[1]}\n")
for tied in [{1: True, 2: True}, {1: False, 2: False}]:   # BOTH tied nearest models
    r = greedy_shrink(tied, xnor, tmgr)
    print(f"  shrink from {tied} -> {fmt(r, tnames):12s} disagreement {disagreement(r, tomega)}")
print("\nboth tied models yield a reason at the true minimum - neither choice is worse")


Delta = (x & y) | (!x & !y),  omega = (x=T, y=F)
nearest distance: 1

  shrink from {1: True, 2: True} -> x & y        disagreement 1
  shrink from {1: False, 2: False} -> !x & !y      disagreement 1

both tied models yield a reason at the true minimum -- neither choice is worse


In [ ]:
# TIE CASE 2: (x & y) | z - tied models leading to structurally DIFFERENT reasons
smgr, (X, Y, Z) = build(3)
psi = (X & Y) | Z
snames = {1: "x", 2: "y", 3: "z"}
somega = {1: True, 2: False, 3: False}

print("Delta = (x & y) | z,  omega = (x=T, y=F, z=F)")
print(f"nearest distance: {nearest_model(psi, somega, 3)[1]}\n")
for tied in [{1: True, 2: False, 3: True}, {1: True, 2: True, 3: False}]:
    r = greedy_shrink(tied, psi, smgr)
    print(f"  shrink from {tied} -> {fmt(r, snames):12s} disagreement {disagreement(r, somega)}")
print("\nthe two reasons are different (z vs x & y) but BOTH are globally minimal")


Delta = (x & y) | z,  omega = (x=T, y=F, z=F)
nearest distance: 1

  shrink from {1: True, 2: False, 3: True} -> z            disagreement 1
  shrink from {1: True, 2: True, 3: False} -> x & y        disagreement 1

the two reasons are different (z vs x & y) but BOTH are globally minimal




## 6. Randomised stress test of Results 1 to 3

 hundreds of random classifiers across random vtree shapes, and for each one check all three results simultaneously against brute force:

1. does the bottom-up distance equal the true minimum distance?
2. is the reconstructed instance a genuine model sitting at exactly that distance?
3. is the shrunk reason a genuine prime implicant achieving the global minimum disagreement, under several random literal-removal orders?



In [ ]:
def random_classifier(mgr, nvars, nclauses, rng):
    """random CNF over nvars variables"""
    f = mgr.true()
    for _ in range(nclauses):
        chosen = rng.sample(range(1, nvars + 1), rng.randint(1, min(3, nvars))) # for each clause It selects between one and three distinct variables
        clause = mgr.false()
        for v in chosen: # for each variable in the clause, randomly choose its polarity (positive or negative) and add it to the clause
            clause = clause | mgr.literal(v if rng.random() < 0.5 else -v)
        f = f & clause
    return f


rng = random.Random(7)
trials = fail_dist = fail_recon = fail_not_pi = fail_not_min = 0

for _ in range(300):

    nvars = rng.randint(3, 6) # 3 to 6 variables
    mgr_r, _ = build(nvars, vtree_type=rng.choice(["right", "balanced", "left"])) # differenr vtree shapes to ensure the algorithm is not sensitive to the vtree structure
    f = random_classifier(mgr_r, nvars, rng.randint(1, 4), rng) # number of clauses between 1 and 4

    if f.is_false():
        continue
    om = {v: rng.random() < 0.5 for v in range(1, nvars + 1)}
    trials += 1

    #  Result 1: distance correct?
    full, d = nearest_model(f, om, nvars)
    _, d_true = brute_nearest(f, om, nvars, mgr_r) # The SDD-based distance is compared against exhaustive enumeration
    if d != d_true:
        fail_dist += 1
        continue

    #  Result 2: reconstruction is a real model at exactly that distance?
    if not is_implicant(full, f, mgr_r) or sum(1 for v in full if full[v] != om[v]) != d: # check that reconstruction:produces a real model and produces one at exactly the claimed distance
        fail_recon += 1
        continue

    #  Result 3: shrink is a genuine PI achieving the global minimum, in ANY order
    pis = all_prime_implicants(f, nvars, mgr_r) #  all prime implicants are enumerated
    if not pis:
        continue
    min_dis = min(disagreement(p, om) for p in pis) # the true minimum disagreement is computed

    for _ in range(4): # four random literal-removal orders are tested
        order = list(range(1, nvars + 1))
        rng.shuffle(order)
        t = greedy_shrink(full, f, mgr_r, order)
        if not any(t == p for p in pis):
            fail_not_pi += 1
        if disagreement(t, om) != min_dis:
            fail_not_min += 1

print(f"random classifiers tested : {trials}")
print(f"shrink orders tested      : {trials * 4}")
print()
print(f"Result 1 - wrong distance                    : {fail_dist}")
print(f"Result 2 - bad reconstruction                : {fail_recon}")
print(f"Result 3 - shrink was not a prime implicant  : {fail_not_pi}")
print(f"Result 3 - shrink was not globally minimal   : {fail_not_min}")
print()
print("PASS" if (fail_dist + fail_recon + fail_not_pi + fail_not_min) == 0 else "FAILURES FOUND")


random classifiers tested : 292
shrink orders tested      : 1168

Result 1 -- wrong distance                    : 0
Result 2 -- bad reconstruction                : 0
Result 3 -- shrink was not a prime implicant  : 0
Result 3 -- shrink was not globally minimal   : 0

PASS




## 7.  bounded menu of alternative reasons





In [ ]:
def recon_with_override(node, omega, memo, override_id, override_idx):
    """Reconstruct as usual, but at one chosen node force a specific (runner-up) branch"""
    if node.is_true():
        return {}
    if node.is_literal():
        lit = node.literal
        return {abs(lit): lit > 0}

    els = list(node.elements())
    if node.id == override_id:
        prime, sub = els[override_idx] # forced runner-up
    else:
        prime, sub, bc = None, None, INF
        for p, s in els:   # usual winner
            cst = node_dist(p, omega, memo) + node_dist(s, omega, memo)
            if cst < bc:
                bc, prime, sub = cst, p, s

    asg = dict(recon_with_override(prime, omega, memo, override_id, override_idx))
    asg.update(recon_with_override(sub, omega, memo, override_id, override_idx))
    return asg


def winning_path(node, omega, memo, acc=None):
    """Collect the decomposition nodes lying on the optimal reconstruction path"""
    if acc is None:
        acc = [] # the list of decomposition choices found so far
    if node.is_true() or node.is_false() or node.is_literal():
        return acc
    els = list(node.elements())
    best_i, best_c, best_e = None, INF, None
    for i, (p, s) in enumerate(els):
        cst = node_dist(p, omega, memo) + node_dist(s, omega, memo)
        if cst < best_c:
            best_c, best_i, best_e = cst, i, (p, s)
    acc.append((node, best_i, best_c, els)) # records not just the winner but also all losers
    winning_path(best_e[0], omega, memo, acc)
    winning_path(best_e[1], omega, memo, acc)
    return acc


def alternatives(node, omega, nvars):
    """ the optimal distance, plus one alternative per losing branch on the path"""
    memo = {}
    d = node_dist(node, omega, memo) # computes the global optimum
    out = []
    for dec, win_i, win_c, els in winning_path(node, omega, memo): # For every decomposition on the winning reconstruction, it examines every losing branch
        for i, (p, s) in enumerate(els):
            if i == win_i:
                continue
            #Its branch cost is:
            cst = node_dist(p, omega, memo) + node_dist(s, omega, memo)
            if cst == INF:   # unsatisfiable branch, not an option
                continue
            partial = recon_with_override(node, omega, memo, dec.id, i)
            full = {v: partial.get(v, omega[v]) for v in range(1, nvars + 1)}
            out.append({
                "gap": cst - win_c, # If the winning branch cost is q and the alternative costs q+g, then g is the gap
                "instance": full,
                "instance_distance": sum(1 for v in full if full[v] != omega[v]),
            })
    return d, out


In [16]:
d, alts = alternatives(phi, omega, 3)
print(f"phi = (a & b) | c,  omega = (F, F, F)")
print(f"optimal correction distance d = {d}\n")

for A in alts:
    r = greedy_shrink(A["instance"], phi, mgr)
    dis = disagreement(r, omega)
    print(f"  ALTERNATIVE  (local gap {A['gap']})")
    print(f"    instance          : {A['instance']}")
    print(f"    its distance      : {A['instance_distance']}   (predicted exactly d+gap = {d + A['gap']})  "
          f"{'OK' if A['instance_distance'] == d + A['gap'] else 'MISMATCH'}")
    print(f"    reason after shrink: {fmt(r, names)}   disagreement {dis}")
    print(f"    bound {d} <= {dis} <= {d + A['gap']}   "
          f"{'satisfied' if d <= dis <= d + A['gap'] else 'VIOLATED'}")
    print(f"    -> shrinking erased the gap entirely: this alternative is just as good as the optimum")


phi = (a & b) | c,  omega = (F, F, F)
optimal correction distance d = 1

  ALTERNATIVE  (local gap 1)
    instance          : {1: True, 2: False, 3: True}
    its distance      : 2   (predicted exactly d+gap = 2)  OK
    reason after shrink: c   disagreement 1
    bound 1 <= 1 <= 2   satisfied
    -> shrinking erased the gap entirely: this alternative is just as good as the optimum


In [17]:
rng = random.Random(11)
n_alts = fail_exact = fail_bound = equally_good = strictly_worse = 0

for _ in range(250):
    nvars = rng.randint(3, 6)
    mgr_r, _ = build(nvars, vtree_type=rng.choice(["right", "balanced"]))
    f = random_classifier(mgr_r, nvars, rng.randint(1, 4), rng)
    if f.is_false():
        continue
    om = {v: rng.random() < 0.5 for v in range(1, nvars + 1)}
    d, alts = alternatives(f, om, nvars)
    if d == INF:
        continue

    for A in alts:
        n_alts += 1
        # EXACT part: alternative instance is a model at distance exactly d + gap
        if A["instance_distance"] != d + A["gap"] or not is_implicant(A["instance"], f, mgr_r):
            fail_exact += 1
        # BOUND part: d <= disagreement(shrunk reason) <= d + gap
        dis = disagreement(greedy_shrink(A["instance"], f, mgr_r), om)
        if not (d <= dis <= d + A["gap"]):
            fail_bound += 1
        if dis == d:
            equally_good += 1
        else:
            strictly_worse += 1

print(f"alternatives generated : {n_alts}\n")
print(f"exact-part failures  (instance is a model at distance exactly d+gap) : {fail_exact}")
print(f"bound-part failures  (d <= disagreement <= d+gap)                    : {fail_bound}")
print()
print(f"  alternatives that shrank back to an equally-good reason : {equally_good}  ({100*equally_good/n_alts:.0f}%)")
print(f"  alternatives that were strictly worse than the optimum  : {strictly_worse}  ({100*strictly_worse/n_alts:.0f}%)")
print()
print("PASS" if (fail_exact + fail_bound) == 0 else "FAILURES FOUND")


alternatives generated : 264

exact-part failures  (instance is a model at distance exactly d+gap) : 0
bound-part failures  (d <= disagreement <= d+gap)                    : 0

  alternatives that shrank back to an equally-good reason : 192  (73%)
  alternatives that were strictly worse than the optimum  : 72  (27%)

PASS




## 8. Scaling: why enumeration is not an option



In [ ]:
rng = random.Random(3)

print(f"{'vars':>5} {'SDD size':>9} {'#reasons':>9} {'enumerate (s)':>14} {'pipeline (s)':>13} {'speedup':>9}")
print("-" * 66)

for nvars in [6, 8, 10, 12, 14, 16]:
    mgr_s, _ = build(nvars, vtree_type="balanced")
    f = satisfiable_classifier(mgr_s, nvars, nvars, rng)
    om = {v: rng.random() < 0.5 for v in range(1, nvars + 1)}

    # the pipeline is executed 200 times, and the average is taken
    REPS = 200
    t0 = time.time()
    for _ in range(REPS): # the pipeline does not include the cost of initially compiling the formula into an SDD
        full, d = nearest_model(f, om, nvars)
        tau = greedy_shrink(full, f, mgr_s)
    t_pipeline = (time.time() - t0) / REPS

    if nvars <= 12:   # brute force becomes infeasible past this
        t0 = time.time()
        pis = all_prime_implicants(f, nvars, mgr_s)
        t_enum = time.time() - t0
        assert disagreement(tau, om) == min(disagreement(p, om) for p in pis)   # still correct
        speedup = f"{t_enum / t_pipeline:>7.0f}x" if t_pipeline > 0 else "     n/a"
        print(f"{nvars:>5} {f.size():>9} {len(pis):>9} {t_enum:>14.3f} {t_pipeline:>13.5f} {speedup:>9}")
    else:
        print(f"{nvars:>5} {f.size():>9} {'?':>9} {'infeasible':>14} {t_pipeline:>13.5f} {'-':>9}")

 vars  SDD size  #reasons  enumerate (s)  pipeline (s)   speedup
------------------------------------------------------------------
    6        31        11          0.034       0.00025      134x
    8       109        27          1.596       0.00039     4040x
   10       156        41          3.581       0.00161     2223x
   12       160        90          9.916       0.00051    19565x
   14       419         ?     infeasible       0.00071         -
   16       837         ?     infeasible       0.00123         -
